# **Phase 4B: Cluster Explainability**
---
**Research question:** Which biomass properties make Cluster 0 different
from Cluster 1 and Cluster 2?

Clusters are fuel typologies, not technologies. **This notebook:**
1. Quantifies per-cluster feature differences (mean / std).
2. Ranks which features best separate the clusters via a Random Forest
   classifier trained to predict cluster membership from explainable,
   decision-relevant indicators.
3. Visualizes each feature's distribution by cluster (boxplots).

In [1]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.analysis.cluster_explainability import run_cluster_explainability

df = pd.read_csv("../data/interim/engineered_features_with_cluster.csv")
df.head()

,Sample_ID,Biomass_Type,Class,Subclass,Ash_db,VM_db,FC_db,C_db,H_db,N_db,...,Energy_Density_Index,Volatile_Fixed_Ratio,Combustibility_Index,Alkali_Index,Silica_Ratio,Base_Acid_Ratio,Moist_ar,Moisture_Penalty,Effective_HHV,Cluster
0,1,Industrial Processing,Timber industry,Woodchips (Softwood),1.1,81.0,17.9,48.6,6.26,0.20,...,367.845,4.525140,334.404545,4.99,0.428990,1.975996,41.2,0.023697,-826.110,1.0
1,3,Agricultural,Animal farming,Chicken manure pellets,32.7,56.2,11.1,30.0,3.91,3.94,...,137.751,5.063063,4.212569,9.03,0.577461,1.623719,18.9,0.050251,-222.139,2.0
2,4,Urban Waste,Biosolids,Treated biosolids,42.8,52.0,5.2,24.7,4.35,4.61,...,65.832,10.000000,1.538131,1.72,6.864608,0.717266,8.1,0.109890,-89.886,2.0
3,5,Industrial Processing,Paper industry,Paper sludge,26.2,64.2,9.6,32.4,4.96,0.47,...,135.744,6.687500,5.181069,0.82,1.640852,0.404991,7.8,0.113636,-96.152,2.0
4,6,Industrial Processing,Cotton Industry,Cotton seed hulls,1.9,77.9,20.2,32.5,6.02,0.45,...,369.256,3.856436,194.345263,37.97,0.331797,2.703448,11.6,0.079365,-193.768,1.0


**`Combustibility_Index`** is intentionally excluded here as it is derived from
features already present (`FC_db`, `CV_MJ/kg_db`, `Ash_db`) and would be
redundant for feature-importance ranking.

In [2]:
results = run_cluster_explainability(
    df,
    tables_dir="../results/tables/phase4b",
    figures_dir="../results/figures/phase4b"
)

results["feature_importance"]

,Feature,Importance
6,Energy_Density_Index,0.300026
2,Silica_Ratio,0.224460
3,Base_Acid_Ratio,0.163489
1,Alkali_Index,0.093893
0,Moisture_Penalty,0.088632
4,Volatile_Fixed_Ratio,0.072260
5,Effective_HHV,0.057241


**Interpretation of feature importance ranking**

Each time a feature helps split samples cleanly into different clusters, it
gains importance; features that repeatedly separate Cluster 0 from 1 or 2
score highest.

**Example:** Biomass samples are primarily differentiated by
energy density (~30% relative importance), followed by ash-chemistry
characteristics such as silica ratio (~22%) and base–acid ratio (~16%).
Secondary influences include alkali-related risk, moisture-induced
performance penalties, and fuel-reactivity indices.

> *"Why is this sample in Cluster 2 and not Cluster 0?"*
> 
> * Because it has lower energy density, higher silica-dominated ash, and 
> 
> * less favorable ash-chemistry balance i.e the strongest discriminating factors identified by the model.

In [3]:
results["cluster_means"]

,Moisture_Penalty,Alkali_Index,Silica_Ratio,Base_Acid_Ratio,Volatile_Fixed_Ratio,Effective_HHV,Energy_Density_Index
Cluster,,,,,,,
0,0.312500,NaN,0.311236,2.530802,108.666667,-48.720000,36.540000
1,0.069803,17.387135,0.517773,16.017476,5.205381,-355.268766,326.713107
2,0.051041,5.369730,21.544669,0.519226,4.995222,-473.254100,151.924500


In [4]:
results["cluster_std"]

,Moisture_Penalty,Alkali_Index,Silica_Ratio,Base_Acid_Ratio,Volatile_Fixed_Ratio,Effective_HHV,Energy_Density_Index
Cluster,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.030921,9.036300,1.084907,22.096074,2.029636,305.918982,83.485052
2,0.041769,3.870473,36.294976,0.554980,2.194084,367.135929,49.318729
